# GlobalCLIP -- 03: Evaluate and Analyse Trained Models

Loads both trained models (Standard and QLayer), evaluates them on the
held-out test set, and generates detailed analysis plots.

**Analyses performed:**
1. Per-sequence Pearson r on log-fold-enrichment (Standard vs QLayer)
2. Protein ranking by mean mixing coefficient α
3. 223×223 α correlation matrix (which proteins co-activate?)
4. Prediction profile comparison (high-signal test examples)
5. QLayer phase polar plot (cooperative vs competitive clusters)
6. QLayer coupling matrix cos(φ_i − φ_j)


## Set-up

### Imports

In [ ]:
import pylbsr.notebooks
import pylbsr.misc

import json
import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt
import yaml
from pathlib import Path
from dotmap import DotMap
from scipy.stats import pearsonr

from parnet_additional_utils import (
    load_parnet_model,
    ParnetModelName,
)
from globalclip_utils import (
    GlobalCLIPStandardModel,
    GlobalCLIPQLayerModel,
    GlobalCLIPDataset,
    collect_alpha,
    rank_proteins,
    alpha_correlation_matrix,
    evaluate_pearson,
    plot_alpha_heatmap,
    plot_top_proteins,
    plot_phase_polar,
    plot_coupling_heatmap,
    plot_pearson_distribution,
    load_run_config,
)


### Initialisation

In [ ]:
_notebook_name = "03_evaluate_and_analyze.py.ipynb"
_notebook_path = f"notebooks/globalclip/{_notebook_name}"

pylbsr.notebooks.enable_cell_timing_metadata(show=True)
logger = pylbsr.misc.init_logger(_notebook_name)
# Walks up from the notebook path until it finds the repo root, so all later
# paths are relative to the project regardless of the Jupyter working dir.
PROJECT_DIR = pylbsr.notebooks.find_project_root_from_notebook_path(_notebook_path)
logger.info(f"Project directory: {PROJECT_DIR}")

### Parameters

In [ ]:
params_gpu_index          = 0

# Run IDs from training notebooks (must match params_run_id set there)
params_standard_run_id    = "globalclip.standard.v1"
params_qlayer_run_id      = "globalclip.qlayer.v1"

# Evaluation
params_batch_size         = 128
params_num_workers        = 4
params_n_profile_examples = 5    # how many example profiles to plot

# Analysis
params_top_n_proteins     = 30   # top proteins to show in bar plot
params_phase_label_thresh = 0.0  # label phases with mean_alpha >= this


### Filepaths and device

In [ ]:
pylbsr.misc.set_seed(42)

device = torch.device(f"cuda:{params_gpu_index}" if torch.cuda.is_available() else "cpu")
if torch.cuda.is_available():
    torch.cuda.set_device(params_gpu_index)
    logger.info(f"GPU: {torch.cuda.get_device_name(device)}")
else:
    logger.warning("No GPU, running on CPU.")

_fp_cfg = yaml.safe_load((PROJECT_DIR / "config" / "filepaths.server.yaml").read_text())
pretrained_model_name = ParnetModelName.PARNET_7M_0_0

def _res(p):
    # filepaths.server.yaml stores paths relative to the project root.
    p = Path(p)
    return p if p.is_absolute() else PROJECT_DIR / p

FILEPATHS = DotMap()
FILEPATHS.pretrained_model    = _res(_fp_cfg["models"][pretrained_model_name.value])
FILEPATHS.standard_run_dir    = PROJECT_DIR / _fp_cfg["results"]["standard_model"] / params_standard_run_id
FILEPATHS.qlayer_run_dir      = PROJECT_DIR / _fp_cfg["results"]["qlayer_model"]   / params_qlayer_run_id
FILEPATHS.analysis_dir        = PROJECT_DIR / _fp_cfg["results"]["analysis"]
FILEPATHS.rbp_names_file      = PROJECT_DIR / "results" / "globalclip" / "datasets" / "rbp_names.txt"
FILEPATHS.analysis_dir.mkdir(parents=True, exist_ok=True)

# Each run's own config records exactly which dataset it was trained on.
# We read the Standard model's dataset and use it for both models below; if
# Standard and QLayer were trained on different datasets this comparison
# would not be a fair one, so keep an eye on run_config.json when picking
# which two run-ids to compare here.
_std_cfg = load_run_config(FILEPATHS.standard_run_dir)
_qlayer_cfg = load_run_config(FILEPATHS.qlayer_run_dir)
FILEPATHS.dataset = Path(_std_cfg["dataset_path"])

for k, v in FILEPATHS.items():
    logger.info(f"{k:30s}: {v}")

## Load RBP names and test data

In [ ]:
rbp_names = FILEPATHS.rbp_names_file.read_text().strip().split("\n")
logger.info(f"Loaded {len(rbp_names)} RBP names.")

test_ds = GlobalCLIPDataset(FILEPATHS.dataset, split="test",
                             seq_len=600, total_key="globalCLIP")
test_loader = torch.utils.data.DataLoader(
    test_ds, batch_size=params_batch_size, shuffle=False,
    num_workers=params_num_workers, pin_memory=torch.cuda.is_available()
)
logger.info(f"Test set: {len(test_ds)} samples")


## Load trained models

**How each extension is meant to improve on the raw PARNET output:**

- **Standard (`MixCoeffHead` + `log_scale`):** PARNET already outputs 223
  independent per-protein binding tracks, but has no notion of *which*
  proteins actually explain the observed GlobalCLIP signal. `log_scale`
  learns a single global amplitude correction per protein (some tracks are
  systematically over/under-scaled by the frozen backbone); `MixCoeffHead`
  additionally learns a *sequence-dependent* weight (α, sigmoid) so the same
  protein can contribute more on some sequences than others. Mechanistically
  this should help whenever the "true" GlobalCLIP signal is a
  sequence-context-dependent subset/weighting of the 223 tracks rather than
  their flat sum.
- **QLayer (adds `QLayer` phase + dilated CNN):** Standard's weighted sum
  cannot represent interactions *between* proteins — each track contributes
  independently. QLayer treats each track as a wave with a learned phase φ_i
  and sums them as complex amplitudes before squaring, which introduces
  pairwise cross-terms `2·α_i·α_j·cos(φ_i−φ_j)`. This lets pairs of proteins
  *reinforce* (φ_i≈φ_j) or *cancel out* (φ_i≈φ_j+π) each other's signal,
  giving the model a cheap (223 extra parameters, not 223²) way to encode
  cooperative/competitive protein relationships. The dilated CNN afterward
  gives the (otherwise position-wise) interference pattern some local
  receptive field to clean up the profile shape. This should only help if
  such interactions actually exist in the data — otherwise it's just extra
  capacity that can overfit (see Section 11).

In [ ]:
logger.info("Loading pretrained PARNET backbone...")
parnet = load_parnet_model(
    pretrained_model_name,
    FILEPATHS.pretrained_model,
    dtype=torch.float32,
    device=device,
)
parnet.eval()

# ── Standard model ────────────────────────────────────────────────────────────
model_std = GlobalCLIPStandardModel(
    parnet_model=parnet,
    num_rbps=_std_cfg["params_num_rbps"],
    mix_hidden=_std_cfg["params_mix_hidden"],
).to(device)
model_std.load_state_dict(
    torch.load(FILEPATHS.standard_run_dir / "model.statedict.pt", map_location=device)
)
model_std.eval()
logger.info(f"Standard model loaded from {FILEPATHS.standard_run_dir}")

# ── QLayer model ──────────────────────────────────────────────────────────────
model_ql = GlobalCLIPQLayerModel(
    parnet_model=parnet,
    num_rbps=_qlayer_cfg["params_num_rbps"],
    mix_hidden=_qlayer_cfg["params_mix_hidden"],
    cnn_channels=_qlayer_cfg["params_cnn_channels"],
    cnn_kernel=_qlayer_cfg["params_cnn_kernel"],
    cnn_layers=_qlayer_cfg["params_cnn_layers"],
).to(device)
model_ql.load_state_dict(
    torch.load(FILEPATHS.qlayer_run_dir / "model.statedict.pt", map_location=device)
)
model_ql.eval()
logger.info(f"QLayer model loaded from {FILEPATHS.qlayer_run_dir}")


## 1 — Test-set Pearson r (Standard vs QLayer)

Measures how well each model's predicted profile *shape* matches
`log1p(signal)` on held-out test sequences (Pearson r — shape only, not
absolute magnitude).

**What to look for:**
- r close to 1.0 = model tracks the true enrichment shape closely; r near 0 =
  no better than noise.
- Compare the *distribution* (median, std, spread), not just the mean — a few
  outlier sequences can skew the mean.
- A model can still be trivial: if the sequence-independent baseline
  (Section 9) scores nearly as high, the model isn't really using sequence
  information — it's just reproducing a shared positional bias.

In [ ]:
logger.info("Evaluating Standard model on test set...")
mean_r_std, all_r_std = evaluate_pearson(model_std, test_loader, device)
logger.info(f"Standard   mean Pearson r = {mean_r_std:.4f}  (std={all_r_std.std():.4f})")

logger.info("Evaluating QLayer model on test set...")
mean_r_ql, all_r_ql = evaluate_pearson(model_ql, test_loader, device)
logger.info(f"QLayer     mean Pearson r = {mean_r_ql:.4f}  (std={all_r_ql.std():.4f})")

print(f"\n{'Model':<20s}  {'Mean Pearson r':>15s}  {'Median':>8s}  {'Std':>8s}")
print("-" * 58)
for name, all_r, mean_r in [("Standard", all_r_std, mean_r_std),
                              ("QLayer",   all_r_ql,  mean_r_ql)]:
    print(f"{name:<20s}  {mean_r:>15.4f}  {np.median(all_r):>8.4f}  {all_r.std():>8.4f}")


In [ ]:
fig = plot_pearson_distribution(all_r_std, all_r_ql)
fig.savefig(FILEPATHS.analysis_dir / "pearson_distribution.png", dpi=120, bbox_inches="tight")
plt.show()

# Save numeric results
results = {
    "standard": {"mean_r": float(mean_r_std), "median_r": float(np.median(all_r_std)), "std_r": float(all_r_std.std())},
    "qlayer":   {"mean_r": float(mean_r_ql),  "median_r": float(np.median(all_r_ql)),  "std_r": float(all_r_ql.std())},
}
(FILEPATHS.analysis_dir / "test_results.json").write_text(json.dumps(results, indent=2))
logger.info(f"Results saved to {FILEPATHS.analysis_dir / 'test_results.json'}")


## 2 — Protein ranking by mixing coefficient

Which RBPs contribute most to the GlobalCLIP signal across all test sequences?

**What to look for:**
- High mean α means the model relies on that protein's track *often* across
  sequences — cross-check top hits against known highly-expressed /
  promiscuous RBPs as a biological plausibility check.
- α alone doesn't capture magnitude: combine with `exp_log_scale` /
  `effective_weight` — a protein with small α but large `log_scale` can still
  dominate the prediction.
- Large `std_alpha` means the protein's importance is sequence-dependent
  (interesting, expected). Near-zero std for *all* proteins suggests the
  MixCoeffHead collapsed to a static, sequence-independent weighting and
  isn't adding value over a fixed linear combination.

In [ ]:
logger.info("Collecting alpha matrices (Standard model)...")
# alpha = the learned, per-sequence mixing weight for each of the 223 RBP
# tracks (see MixCoeffHead in model.py); averaging it over the test set and
# ranking by the mean is what produces the protein ranking below.
alpha_std = collect_alpha(model_std, test_loader, device)
logger.info(f"alpha_std shape: {alpha_std.shape}")

log_scale_std = model_std.log_scale.exp().detach().cpu().numpy()
ranking_std = rank_proteins(alpha_std, rbp_names, log_scale=log_scale_std)

print("Top-20 proteins (Standard model):")
print(ranking_std.head(20).to_string(index=False))

In [ ]:
fig = plot_top_proteins(ranking_std, top_n=params_top_n_proteins)
fig.savefig(FILEPATHS.analysis_dir / "protein_ranking_standard.png", dpi=120, bbox_inches="tight")
plt.show()

ranking_std.to_csv(FILEPATHS.analysis_dir / "protein_ranking_standard.tsv", sep="\t", index=False)


## 3 — Alpha correlation matrix

Proteins with correlated mixing coefficients tend to co-activate on the same
sequences — they likely belong to the same RNP complex or bind related
motifs.

**What to look for:**
- Strongly red blocks (r ≈ +1) = protein pairs whose usage co-varies across
  sequences — likely co-regulated or partially redundant PARNET tracks, not
  necessarily a real physical interaction.
- This only reflects correlation in *predicted* α, not measured protein-protein
  binding — treat clusters as hypothesis-generating, not confirmatory.

In [ ]:
corr_std = alpha_correlation_matrix(alpha_std)

fig = plot_alpha_heatmap(corr_std, rbp_names=None)   # too many names for labels
fig.savefig(FILEPATHS.analysis_dir / "alpha_correlation_standard.png", dpi=150, bbox_inches="tight")
plt.show()

# Top-10 most correlated pairs (excluding self)
idx = np.triu_indices(223, k=1)
pairs = [(corr_std[i, j], rbp_names[i], rbp_names[j])
         for i, j in zip(idx[0], idx[1])]
pairs.sort(key=lambda x: abs(x[0]), reverse=True)
print("Top-10 most correlated protein pairs (Standard model):")
print(f"{'r':>7s}  {'Protein A':<25s}  {'Protein B'}")
print("-" * 65)
for r, a, b in pairs[:10]:
    print(f"{r:>7.3f}  {a:<25s}  {b}")


## 4 — Prediction profiles (high-signal examples)

Select the test sequences with the highest total GlobalCLIP signal and compare
the ground truth log-FE against both model predictions.

**What to look for:**
- Compare peak *positions and shapes*, not just overall visual overlap — do
  predicted peaks line up with real peaks, or is the model just producing a
  smoothed version of the average profile?
- If both models produce near-identical, blurry profiles regardless of the
  true signal's shape, they may be relying mostly on shared positional bias
  rather than sequence-specific signal (compare against Section 9's baseline).

In [ ]:
# Collect all signal totals to find high-signal sequences
from globalclip_utils.datasets import GlobalCLIPDataset

signal_totals = []
with torch.no_grad():
    for batch in test_loader:
        signal_totals.extend(batch["signal"].squeeze(1).sum(-1).tolist())
signal_totals = np.array(signal_totals)
top_indices = signal_totals.argsort()[::-1][:params_n_profile_examples]
logger.info(f"Top-{params_n_profile_examples} signal totals: {signal_totals[top_indices]}")


In [ ]:
# Three columns per row: ground-truth signal, Standard prediction, QLayer
# prediction, for the highest-signal test sequences (loudest/clearest
# examples, easiest to visually judge whether shape is being captured).
fig, axes = plt.subplots(params_n_profile_examples, 3,
                          figsize=(16, 3.5 * params_n_profile_examples),
                          sharex=True)

with torch.no_grad():
    for row, idx in enumerate(top_indices):
        sample    = test_ds[idx]
        seq       = sample["sequence"].unsqueeze(0).to(device)
        signal    = sample["signal"]

        target    = torch.log1p(signal).squeeze().numpy()
        signal_np = signal.squeeze().numpy()

        pred_std, _ = model_std(seq)
        pred_ql,  _ = model_ql(seq)
        pred_std_np = pred_std.squeeze().cpu().numpy()
        pred_ql_np  = pred_ql.squeeze().cpu().numpy()

        r_std, _ = pearsonr(pred_std_np, target)
        r_ql,  _ = pearsonr(pred_ql_np,  target)

        pos = np.arange(600)
        kw  = dict(linewidth=0, alpha=0.8)

        axes[row, 0].fill_between(pos, signal_np, color="black", **kw)
        axes[row, 0].set_ylabel(f"Sample {idx}\ncounts", fontsize=8)
        if row == 0:
            axes[row, 0].set_title("Ground truth (GlobalCLIP signal)")

        axes[row, 1].fill_between(pos, pred_std_np, color="steelblue", **kw)
        axes[row, 1].set_ylabel(f"r={r_std:.3f}", fontsize=8, color="steelblue")
        if row == 0:
            axes[row, 1].set_title("Standard model prediction")

        axes[row, 2].fill_between(pos, pred_ql_np, color="darkorange", **kw)
        axes[row, 2].set_ylabel(f"r={r_ql:.3f}", fontsize=8, color="darkorange")
        if row == 0:
            axes[row, 2].set_title("QLayer model prediction")

plt.suptitle("Prediction profiles on high-signal test sequences", y=1.01)
plt.tight_layout()
plt.savefig(FILEPATHS.analysis_dir / "profile_comparison.png", dpi=120, bbox_inches="tight")
plt.show()

## 5 — QLayer phase analysis

After training, each of the 223 proteins has a learned phase φ_i.
Proteins clustered at similar phases cooperate (constructive interference);
proteins at opposite phases compete or cancel (destructive interference —
the mechanism by which the QLayer suppresses background noise).

**What to look for:**
- Phases spread across the circle with clear clusters = the model found
  meaningful cooperative/competitive groups.
- All phases collapsed near a single value = the interference term degenerates
  to a constant scale factor — the QLayer isn't adding anything beyond
  Standard, and the CNN afterward is doing all the work.
- Cross-check: do the high-α proteins from Section 2 cluster together in
  phase, or are they scattered?

In [ ]:
logger.info("Collecting alpha for QLayer model...")
alpha_ql = collect_alpha(model_ql, test_loader, device)
ranking_ql = rank_proteins(alpha_ql, rbp_names)

phases = model_ql.qlayer.phase.detach().cpu().numpy()
mean_alpha_ql = alpha_ql.mean(0)

print(f"Phase range: [{phases.min():.3f}, {phases.max():.3f}] rad")
print(f"Phase std:   {phases.std():.3f} rad")

# Label the most important proteins
params_phase_label_thresh = float(np.percentile(mean_alpha_ql, 80))
print(f"Labeling proteins with mean_alpha >= {params_phase_label_thresh:.4f} (top 20%)")


In [ ]:
fig = plot_phase_polar(
    phases=phases,
    rbp_names=rbp_names,
    label_threshold=params_phase_label_thresh,
    alpha_values=mean_alpha_ql,
)
fig.savefig(FILEPATHS.analysis_dir / "qlayer_phase_polar.png", dpi=150, bbox_inches="tight")
plt.show()


In [ ]:
# Phase-sorted protein table
phase_df = pd.DataFrame({
    "protein":    rbp_names,
    "phase_rad":  phases,
    "mean_alpha": mean_alpha_ql,
}).sort_values("phase_rad").reset_index(drop=True)

print("Proteins sorted by phase (head = most constructive, tail = most destructive):")
print(phase_df[["protein","phase_rad","mean_alpha"]].head(10).to_string(index=False))
print("...")
print(phase_df[["protein","phase_rad","mean_alpha"]].tail(10).to_string(index=False))

phase_df.to_csv(FILEPATHS.analysis_dir / "qlayer_phases.tsv", sep="\t", index=False)


## 6 — QLayer coupling matrix

`cos(φ_i − φ_j)`: +1 = fully cooperative, −1 = fully competitive.

**What to look for:**
- This matrix depends only on *learned phases*, not on how often a pair
  actually co-occurs (α) — a strongly "competitive" pair (cos ≈ −1) between
  two rarely-active proteins is not biologically meaningful. Always
  cross-reference with `mean_alpha` from Section 5.
- A mostly uniform matrix (little red/blue contrast) suggests the phase
  mechanism hasn't learned much structure yet.

In [ ]:
coupling = model_ql.get_coupling_matrix().cpu().numpy()

fig = plot_coupling_heatmap(coupling)
fig.savefig(FILEPATHS.analysis_dir / "qlayer_coupling.png", dpi=150, bbox_inches="tight")
plt.show()

# Top cooperative pairs
idx = np.triu_indices(223, k=1)
coup_pairs = [(coupling[i, j], rbp_names[i], rbp_names[j])
              for i, j in zip(idx[0], idx[1])]
coup_pairs.sort(key=lambda x: x[0], reverse=True)

print("Top-10 cooperative protein pairs (cos(φ_i−φ_j) ≈ 1):")
for c, a, b in coup_pairs[:10]:
    print(f"  cos(Δφ)={c:+.3f}  {a}  ↔  {b}")

print("\nTop-10 competitive protein pairs (cos(φ_i−φ_j) ≈ −1):")
for c, a, b in coup_pairs[-10:][::-1]:
    print(f"  cos(Δφ)={c:+.3f}  {a}  ↔  {b}")


## 7 — QLayer alpha correlation (comparison with Standard)

Side-by-side comparison of which proteins the two architectures rely on
jointly.

**What to look for:**
- Similar correlation structure between Standard and QLayer = both models
  converge on the same underlying protein groupings — reassuring, the
  signal is robust to architecture choice.
- Very different structure = one architecture may be fitting different
  (possibly spurious) patterns; investigate before trusting either model's
  biological story.

In [ ]:
corr_ql = alpha_correlation_matrix(alpha_ql)

fig, axes = plt.subplots(1, 2, figsize=(18, 7))
for ax, corr, name in [
    (axes[0], corr_std, "Standard model"),
    (axes[1], corr_ql,  "QLayer model"),
]:
    im = ax.imshow(corr, cmap="RdBu_r", vmin=-1, vmax=1, aspect="auto")
    plt.colorbar(im, ax=ax, shrink=0.8)
    ax.set_title(f"Alpha correlation matrix\n{name}")

plt.tight_layout()
plt.savefig(FILEPATHS.analysis_dir / "alpha_correlation_comparison.png", dpi=120, bbox_inches="tight")
plt.show()


## Summary table

In [ ]:
summary = pd.DataFrame([
    {
        "Model":           "Standard (MixCoeffHead + log_scale)",
        "Trainable params": sum(p.numel() for p in model_std.parameters() if p.requires_grad),
        "Mean Pearson r":   f"{mean_r_std:.4f}",
        "Median Pearson r": f"{np.median(all_r_std):.4f}",
    },
    {
        "Model":           "QLayer (QLayer + dilated CNN)",
        "Trainable params": sum(p.numel() for p in model_ql.parameters() if p.requires_grad),
        "Mean Pearson r":   f"{mean_r_ql:.4f}",
        "Median Pearson r": f"{np.median(all_r_ql):.4f}",
    },
])

print("\n=== EVALUATION SUMMARY ===")
print(summary.to_string(index=False))
print("\nAll plots saved to:", FILEPATHS.analysis_dir)


## 8 — Naive baseline & statistical significance

Is the learned combination (Standard / QLayer) actually better than "the
original model" — i.e. the frozen PARNET backbone's 223 raw RBP tracks with
no learned weighting at all? We compare against a naive baseline (uniform
mean of the raw tracks) and test whether any improvement is statistically
significant, not just a difference in the mean.

**What follows:**
1. A naive, non-learned baseline (uniform mean of the raw PARNET tracks)
   evaluated with the identical metric.
2. Paired Wilcoxon signed-rank tests — since the same test sequences are
   scored by every model, we pair the per-sequence r values and ask whether
   one model beats another *consistently*, not just on average.
3. Bootstrap 95% confidence intervals on the mean paired difference — if the
   interval excludes 0, the difference is unlikely to be sampling noise from
   this particular test set.

**What to look for:**
- A tiny mean-r improvement with p < 0.05 and a CI excluding 0 is a real (if
  small) effect. A larger improvement with p > 0.05 is *not* trustworthy —
  it could easily reverse on a different test split.
- If Standard barely beats the naive baseline, the learned MixCoeffHead isn't
  adding much value yet. If QLayer barely beats Standard, the phase/CNN
  machinery isn't paying for its extra complexity — see the capacity-matched
  ablation note at the very end of the notebook.

In [ ]:
import torch.nn as nn


class NaiveBaselineModel(nn.Module):
    """Un-learned baseline: uniform-weighted mean of the 223 raw PARNET RBP tracks.

    Represents "the original model" with no GlobalCLIP combination layer at
    all -- same frozen backbone, but no learned alpha / log_scale / QLayer.
    """

    def __init__(self, parnet_model: nn.Module, num_rbps: int = 223):
        super().__init__()
        self.backbone = parnet_model
        self.num_rbps = num_rbps

    @torch.no_grad()
    def forward(self, seq_onehot: torch.Tensor):
        x = self.backbone.stem(seq_onehot)
        x = self.backbone.body(x)
        if hasattr(self.backbone, "projection"):
            x = self.backbone.projection(x)
        rbp_tracks = self.backbone.head.head_target.pointwise_conv(x)   # (B, 223, L)
        pred = rbp_tracks.mean(dim=1, keepdim=True)                     # (B, 1, L)
        alpha = torch.full(
            (seq_onehot.shape[0], self.num_rbps),
            1.0 / self.num_rbps,
            device=seq_onehot.device,
        )
        return pred, alpha


model_base = NaiveBaselineModel(parnet, num_rbps=223).to(device)

logger.info("Evaluating naive baseline (uniform mean of raw RBP tracks) on test set...")
mean_r_base, all_r_base = evaluate_pearson(model_base, test_loader, device)
logger.info(f"Baseline   mean Pearson r = {mean_r_base:.4f}  (std={all_r_base.std():.4f})")

print(f"\n{'Model':<20s}  {'Mean Pearson r':>15s}  {'Median':>8s}  {'Std':>8s}")
print("-" * 58)
for name, all_r, mean_r in [
    ("Baseline (naive)", all_r_base, mean_r_base),
    ("Standard",         all_r_std,  mean_r_std),
    ("QLayer",           all_r_ql,   mean_r_ql),
]:
    print(f"{name:<20s}  {mean_r:>15.4f}  {np.median(all_r):>8.4f}  {all_r.std():>8.4f}")

In [ ]:
from scipy.stats import wilcoxon


def paired_significance(name_a, r_a, name_b, r_b):
    """Paired Wilcoxon signed-rank test on per-sequence Pearson r (same test set)."""
    stat, p = wilcoxon(r_b, r_a)
    diff = r_b.mean() - r_a.mean()
    print(f"{name_b:<10s} vs {name_a:<18s}: mean Δr = {diff:+.4f}   Wilcoxon p = {p:.2e}")
    return p


print("=== Paired significance tests (per-sequence Pearson r, same test sequences) ===")
p_base_std = paired_significance("Baseline", all_r_base, "Standard", all_r_std)
p_base_ql  = paired_significance("Baseline", all_r_base, "QLayer",   all_r_ql)
p_std_ql   = paired_significance("Standard", all_r_std,  "QLayer",   all_r_ql)

In [ ]:
def bootstrap_mean_diff_ci(r_a, r_b, n_boot=10000, seed=42):
    """95% bootstrap CI for the mean paired difference (r_b - r_a)."""
    rng = np.random.default_rng(seed)
    diffs = r_b - r_a
    n = len(diffs)
    idx = rng.integers(0, n, size=(n_boot, n))
    boot_means = diffs[idx].mean(axis=1)
    lo, hi = np.percentile(boot_means, [2.5, 97.5])
    return diffs.mean(), lo, hi


print("=== Bootstrap 95% CI for mean Δr (10000 resamples) ===")
for name_a, r_a, name_b, r_b in [
    ("Baseline", all_r_base, "Standard", all_r_std),
    ("Baseline", all_r_base, "QLayer",   all_r_ql),
    ("Standard", all_r_std,  "QLayer",   all_r_ql),
]:
    mean_diff, lo, hi = bootstrap_mean_diff_ci(r_a, r_b)
    sig = "significant" if lo > 0 or hi < 0 else "not significant"
    print(f"{name_b:<10s} - {name_a:<10s}: Δr = {mean_diff:+.4f}  95% CI [{lo:+.4f}, {hi:+.4f}]  ({sig})")

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))
kw = dict(bins=50, alpha=0.6, edgecolor="none")
for all_r, label, color in [
    (all_r_base, f"Baseline (mean={mean_r_base:.3f})", "gray"),
    (all_r_std,  f"Standard (mean={mean_r_std:.3f})",  "steelblue"),
    (all_r_ql,   f"QLayer   (mean={mean_r_ql:.3f})",   "darkorange"),
]:
    ax.hist(all_r, label=label, color=color, **kw)
    ax.axvline(all_r.mean(), color=color, linestyle="--", linewidth=1.5)
ax.set_xlabel("Pearson r (pred vs. log1p(signal))")
ax.set_ylabel("Number of sequences")
ax.set_title("Test-set Pearson r: baseline vs. learned combinations")
ax.legend()
plt.tight_layout()
fig.savefig(FILEPATHS.analysis_dir / "pearson_distribution_with_baseline.png", dpi=120, bbox_inches="tight")
plt.show()

# Update saved results with baseline + significance
results_with_baseline = {
    "baseline": {"mean_r": float(mean_r_base), "median_r": float(np.median(all_r_base)), "std_r": float(all_r_base.std())},
    "standard": {"mean_r": float(mean_r_std),  "median_r": float(np.median(all_r_std)),  "std_r": float(all_r_std.std())},
    "qlayer":   {"mean_r": float(mean_r_ql),   "median_r": float(np.median(all_r_ql)),   "std_r": float(all_r_ql.std())},
    "significance_wilcoxon_p": {
        "standard_vs_baseline": float(p_base_std),
        "qlayer_vs_baseline":   float(p_base_ql),
        "qlayer_vs_standard":   float(p_std_ql),
    },
}
(FILEPATHS.analysis_dir / "test_results.json").write_text(json.dumps(results_with_baseline, indent=2))
logger.info(f"Updated results (with baseline + significance) saved to {FILEPATHS.analysis_dir / 'test_results.json'}")

## 9 — Sequence-independent baseline (mean training profile)

Many CLIP-seq / enrichment tasks have a strong **positional bias** shared
across almost all sequences (library-prep artifacts, mappability, average
binding density by position). A model can score a deceptively high Pearson r
just by reproducing this shared shape, without using the input sequence at
all.

This cell computes the *average* `log1p(signal)` profile over the training
set (a single length-600 vector, entirely ignoring sequence identity) and
scores it against every test sequence with the same Pearson r metric used
above.

**What to look for:**
- If this sequence-independent baseline scores much lower than
  Standard/QLayer, the learned models are genuinely using sequence
  information — good.
- If it scores close to Standard/QLayer, most of the apparent "predictive
  power" is just the shared positional bias, and the sequence-dependent
  machinery (MixCoeffHead, QLayer) isn't contributing much beyond it.

In [ ]:
logger.info("Computing sequence-independent mean training profile...")
train_ds_profile = GlobalCLIPDataset(FILEPATHS.dataset, split="train", seq_len=600, total_key="globalCLIP")
train_loader_profile = torch.utils.data.DataLoader(
    train_ds_profile, batch_size=256, shuffle=False, num_workers=params_num_workers,
)

profile_sum = torch.zeros(600)
n_train = 0
for batch in train_loader_profile:
    target = torch.log1p(batch["signal"]).squeeze(1)   # (B, L)
    profile_sum += target.sum(0)
    n_train += target.shape[0]
mean_train_profile = (profile_sum / n_train).numpy()     # (L,)
logger.info(f"Mean training profile computed over {n_train} sequences.")

# Score the flat mean profile against every test sequence
mp_tensor = torch.from_numpy(mean_train_profile)
mp_z = mp_tensor - mp_tensor.mean()
mp_norm = mp_z.norm()

all_r_meanprofile = []
for batch in test_loader:
    target = torch.log1p(batch["signal"]).squeeze(1)      # (B, L)
    tz = target - target.mean(-1, keepdim=True)
    r = (tz * mp_z[None, :]).sum(-1) / (tz.norm(dim=-1) * mp_norm + 1e-8)
    all_r_meanprofile.extend(r.numpy().tolist())
all_r_meanprofile = np.array(all_r_meanprofile)
mean_r_meanprofile = float(all_r_meanprofile.mean())

logger.info(f"Sequence-independent baseline mean Pearson r = {mean_r_meanprofile:.4f}")
print(f"\n{'Model':<28s}  {'Mean Pearson r':>15s}")
print("-" * 48)
for name, mean_r in [
    ("Mean-profile (no sequence)",   mean_r_meanprofile),
    ("Naive baseline (mean tracks)", mean_r_base),
    ("Standard",                     mean_r_std),
    ("QLayer",                       mean_r_ql),
]:
    print(f"{name:<28s}  {mean_r:>15.4f}")

## 10 — Best single raw RBP track

The frozen PARNET backbone already outputs 223 individual per-protein
binding tracks — this *is* "the original model" before any GlobalCLIP-specific
combination step. Here we check, on a subsample of the test set, how well the
single best-correlating raw track alone predicts the GlobalCLIP signal.

**What to look for:**
- If one individual track already comes close to Standard/QLayer's mean r,
  the learned combination is mostly just *selecting* a dominant protein
  rather than learning a genuinely new, more informative signal.
- A meaningful gap (Standard/QLayer clearly ahead of the best single track)
  shows real added value from combining multiple proteins.

In [ ]:
logger.info("Computing per-track Pearson r on a subsample of the test set...")
max_batches_track = 20   # subsample for compute, roughly 20*batch_size sequences

# This deliberately bypasses GlobalCLIPStandardModel / _extract_parnet_features
# and re-runs the frozen backbone manually, because we want each of the 223
# raw PARNET tracks' correlation with the target INDIVIDUALLY, before any
# learned combination -- i.e. "how good is the single best pre-trained track
# on its own", as a sanity floor: if the learned mixture cannot beat this,
# the combination layer is not adding value over just picking one track.
track_r_sum = torch.zeros(223)
n_track_batches = 0
with torch.no_grad():
    for i, batch in enumerate(test_loader):
        if i >= max_batches_track:
            break
        seq    = batch["sequence"].to(device)
        signal = batch["signal"].to(device)

        x = parnet.stem(seq)
        x = parnet.body(x)
        if hasattr(parnet, "projection"):
            x = parnet.projection(x)
        rbp_tracks = parnet.head.head_target.pointwise_conv(x)   # (B, 223, L)

        target = torch.log1p(signal)                              # (B, 1, L)
        p = rbp_tracks - rbp_tracks.mean(-1, keepdim=True)
        t = target - target.mean(-1, keepdim=True)
        r = (p * t).sum(-1) / (p.norm(dim=-1) * t.norm(dim=-1) + 1e-8)   # (B, 223)
        track_r_sum += r.mean(0).cpu()
        n_track_batches += 1

mean_track_r = (track_r_sum / n_track_batches).numpy()
best_idx = np.argsort(mean_track_r)[::-1][:5]

print("Top-5 individual RBP tracks by test-set Pearson r:")
for idx in best_idx:
    print(f"  {rbp_names[idx]:<25s}  r = {mean_track_r[idx]:.4f}")

print(f"\nBest single track       : r = {mean_track_r[best_idx[0]]:.4f}  ({rbp_names[best_idx[0]]})")
print(f"Standard (learned mix)  : r = {mean_r_std:.4f}")
print(f"QLayer (learned mix)    : r = {mean_r_ql:.4f}")

## 11 — Train vs. test Pearson r (overfitting check)

Evaluates Standard and QLayer on a subsample of the *training* set with the
same metric as Section 1, to compare directly against the test-set numbers.

**What to look for:**
- A small train/test gap (a fraction of a Pearson-r point) is normal and
  expected.
- A large gap (train r ≫ test r) means the model is overfitting to the
  training sequences. With only a frozen-backbone MLP head this is less
  likely than for a full fine-tune, but QLayer's extra CNN + phase
  parameters give it more room to overfit than the simpler Standard model —
  worth checking specifically for QLayer.

In [ ]:
logger.info("Building a training-set subsample loader for the overfitting check...")
train_ds_eval = GlobalCLIPDataset(FILEPATHS.dataset, split="train", seq_len=600, total_key="globalCLIP")
train_loader_eval = torch.utils.data.DataLoader(
    train_ds_eval, batch_size=params_batch_size, shuffle=True,
    num_workers=params_num_workers, pin_memory=torch.cuda.is_available(),
)

max_batches_overfit = 20   # subsample for compute


def evaluate_pearson_subset(model, dataloader, device, max_batches):
    # Same per-sequence Pearson computation as evaluate_pearson in
    # analysis_utils.py, just capped at max_batches so this stays cheap
    # (it is only meant to answer "is train Pearson notably above test
    # Pearson", not to be an exact number).
    model.eval()
    corrs = []
    with torch.no_grad():
        for i, batch in enumerate(dataloader):
            if i >= max_batches:
                break
            seq    = batch["sequence"].to(device)
            signal = batch["signal"].to(device)
            pred, _ = model(seq)
            target = torch.log1p(signal)
            p = pred.squeeze(1); t = target.squeeze(1)
            pz = p - p.mean(-1, keepdim=True); tz = t - t.mean(-1, keepdim=True)
            r = (pz * tz).sum(-1) / (pz.norm(dim=-1) * tz.norm(dim=-1) + 1e-8)
            corrs.extend(r.cpu().float().numpy().tolist())
    return float(np.mean(corrs)), np.array(corrs)


mean_r_std_train, _ = evaluate_pearson_subset(model_std, train_loader_eval, device, max_batches_overfit)
mean_r_ql_train,  _ = evaluate_pearson_subset(model_ql,  train_loader_eval, device, max_batches_overfit)

# A large positive gap (train notably above test) would suggest the
# combination layer is memorising training positions rather than learning
# a genuinely transferable mixture; a small gap argues against that.
print(f"{'Model':<12s}  {'Train r':>10s}  {'Test r':>10s}  {'Gap':>8s}")
print("-" * 46)
print(f"{'Standard':<12s}  {mean_r_std_train:>10.4f}  {mean_r_std:>10.4f}  {mean_r_std_train - mean_r_std:>+8.4f}")
print(f"{'QLayer':<12s}  {mean_r_ql_train:>10.4f}  {mean_r_ql:>10.4f}  {mean_r_ql_train - mean_r_ql:>+8.4f}")

## 12 — Training dynamics: train vs. validation loss across epochs

Loads each run's Lightning `CSVLogger` output
(`csv_logs/version_*/metrics.csv`) to compare `train/*` vs `val/*` curves
epoch-by-epoch for both models — total loss, Pearson loss, NLL, and the
alpha sparsity penalty.

**What to look for:**
- Val loss should decrease then flatten; if it starts *increasing* while
  train loss keeps dropping, that's the classic overfitting signature —
  check where `EarlyStopping` (`patience=8` by default) actually stopped
  training relative to that point.
- A widening train/val gap over epochs = overfitting; a persistent,
  roughly-constant gap is normal.
- Compare convergence *speed* between Standard and QLayer — QLayer's phases
  train at 10× the base learning rate (see `configure_optimizers` in
  `training_utils.py`), so watch whether it converges faster/noisier or
  needs more epochs to stabilize.
- If a run stopped well before `--max-epochs 50`, `EarlyStopping` triggered
  — val loss had plateaued for `patience` epochs.

In [ ]:
def load_metrics_epoch_df(run_dir):
    csv_log_dir = run_dir / "csv_logs"
    # Lightning starts a new version_N/ folder each time training is
    # restarted in this output dir, so always take the latest one.
    metrics_paths = sorted(csv_log_dir.glob("version_*/metrics.csv"))
    if not metrics_paths:
        raise FileNotFoundError(f"No csv_logs/version_*/metrics.csv found under {csv_log_dir}")
    df = pd.read_csv(metrics_paths[-1])
    # Keep only the last logged row per epoch, drops intra-epoch step noise.
    return df.groupby("epoch").last().reset_index()


epoch_df_std = load_metrics_epoch_df(FILEPATHS.standard_run_dir)
epoch_df_ql  = load_metrics_epoch_df(FILEPATHS.qlayer_run_dir)

metrics_to_plot = ["loss", "pearson", "nll", "alpha_mean"]

fig, axes = plt.subplots(2, len(metrics_to_plot), figsize=(4 * len(metrics_to_plot), 7), sharex=True)
for row, (epoch_df, name) in enumerate([(epoch_df_std, "Standard"), (epoch_df_ql, "QLayer")]):
    for col, metric in enumerate(metrics_to_plot):
        ax = axes[row, col]
        train_col = f"train/{metric}_epoch"
        val_col   = f"val/{metric}"
        if train_col in epoch_df.columns:
            ax.plot(epoch_df["epoch"], epoch_df[train_col], marker="o", markersize=3, label="train")
        if val_col in epoch_df.columns:
            ax.plot(epoch_df["epoch"], epoch_df[val_col], marker="o", markersize=3, label="val")
        ax.set_title(f"{name} - {metric}")
        ax.set_xlabel("epoch")
        ax.legend(fontsize=7)

plt.tight_layout()
fig.savefig(FILEPATHS.analysis_dir / "train_val_curves_comparison.png", dpi=120, bbox_inches="tight")
plt.show()

# Val loss below train loss here is expected (dropout/batchnorm are active
# in training but disabled at eval), not itself a sign of overfitting -- the
# thing to actually watch for is the gap widening over many epochs.
print(f"{'Model':<10s}  {'Epochs run':>10s}  {'Best val/loss':>14s}  {'Final train/loss':>17s}  {'Gap (train-val)':>16s}")
print("-" * 74)
for epoch_df, name in [(epoch_df_std, "Standard"), (epoch_df_ql, "QLayer")]:
    n_epochs = int(epoch_df["epoch"].max()) + 1
    best_val = epoch_df["val/loss"].min() if "val/loss" in epoch_df.columns else float("nan")
    final_train = epoch_df["train/loss_epoch"].iloc[-1] if "train/loss_epoch" in epoch_df.columns else float("nan")
    gap = final_train - best_val
    print(f"{name:<10s}  {n_epochs:>10d}  {best_val:>14.4f}  {final_train:>17.4f}  {gap:>+16.4f}")

## 13 — Spearman rank correlation (robustness check)

Pearson r assumes a roughly *linear* relationship between prediction and
target; Spearman only requires a *monotonic* one (rank correlation). Signal
profiles are noisy counts with many ties (lots of near-zero positions), so
comparing both tells us whether the models are capturing genuine ranking of
"high vs low" positions or are only fitting a specific linear scale that
Pearson happens to reward.

**What to look for:**
- Pearson and Spearman should broadly agree in *ranking the three models*
  (baseline < Standard < QLayer, or whatever order Section 1/8 found). A
  large disagreement (e.g. QLayer wins on Pearson but loses on Spearman)
  suggests its apparent edge comes from fitting magnitude/scale rather than
  genuinely better localization of signal.
- Report both **mean** and **median** for each metric: profile-level
  Pearson/Spearman values are bounded in [-1, 1] and can be skewed by a few
  low-signal sequences where any correlation is noisy — the median is a more
  robust summary than the mean in that case.
- Note: ties (very common in sparse count data) are broken arbitrarily by
  the rank transform used here rather than averaged — a standard
  simplification, but it means Spearman values for very sparse profiles
  should be read as approximate.

In [ ]:
def _rank_last_dim(x: torch.Tensor) -> torch.Tensor:
    """Rank-transform along the last dim (ties broken arbitrarily, not averaged)."""
    order = x.argsort(dim=-1)
    ranks = torch.empty_like(order, dtype=torch.float32)
    arange = torch.arange(x.shape[-1], dtype=torch.float32, device=x.device).expand_as(x)
    ranks.scatter_(-1, order, arange)
    return ranks


@torch.no_grad()
def evaluate_spearman(model, dataloader, device):
    """Per-sequence Spearman rank correlation (pred vs. log1p(signal))."""
    model.eval()
    corrs = []
    for batch in dataloader:
        seq    = batch["sequence"].to(device)
        signal = batch["signal"].to(device)
        pred, _ = model(seq)
        target = torch.log1p(signal)

        p = _rank_last_dim(pred.squeeze(1))
        t = _rank_last_dim(target.squeeze(1))
        pz = p - p.mean(-1, keepdim=True)
        tz = t - t.mean(-1, keepdim=True)
        r = (pz * tz).sum(-1) / (pz.norm(dim=-1) * tz.norm(dim=-1) + 1e-8)
        corrs.extend(r.cpu().float().numpy().tolist())
    all_r = np.array(corrs)
    return float(np.mean(all_r)), all_r


logger.info("Evaluating Spearman rank correlation on test set...")
mean_rho_base, all_rho_base = evaluate_spearman(model_base, test_loader, device)
mean_rho_std,  all_rho_std  = evaluate_spearman(model_std,  test_loader, device)
mean_rho_ql,   all_rho_ql   = evaluate_spearman(model_ql,   test_loader, device)

print(f"{'Model':<20s}  {'Pearson mean':>13s}  {'Pearson med.':>13s}  {'Spearman mean':>14s}  {'Spearman med.':>14s}")
print("-" * 82)
for name, r_arr, rho_arr in [
    ("Baseline (naive)", all_r_base, all_rho_base),
    ("Standard",         all_r_std,  all_rho_std),
    ("QLayer",           all_r_ql,   all_rho_ql),
]:
    print(f"{name:<20s}  {r_arr.mean():>13.4f}  {np.median(r_arr):>13.4f}  "
          f"{rho_arr.mean():>14.4f}  {np.median(rho_arr):>14.4f}")

## Overall interpretation guide

Putting Sections 1–13 together, to decide whether the extra architecture is
worth it:

1. **Sanity floor** (Section 9): both models should clearly beat the
   sequence-independent mean-profile baseline. If not, they're not using
   sequence information at all.
2. **Value of learning a combination** (Section 10): Standard/QLayer should
   clearly beat the best single raw track. If not, the MixCoeffHead isn't
   adding value beyond picking one dominant protein.
3. **Statistical reality of any gain** (Section 8): trust a mean-r
   improvement only if the paired Wilcoxon p-value is small and the
   bootstrap CI excludes 0.
4. **Overfitting** — both the post-hoc train/test r gap (Section 11) and the
   actual per-epoch train/val loss curves (Section 12) should be checked
   together; a model with a much larger gap should be trusted less even if
   its test r is nominally higher.
5. **Rank vs. linear fit** (Section 13): Pearson and Spearman should agree on
   which model is best; a mismatch signals scale-fitting rather than real
   localization improvement.
6. **QLayer's added complexity** (Sections 5–7): only interpret the
   phase/coupling story as biologically meaningful if QLayer clearly beats
   Standard on (1)–(5) above — otherwise the extra parameters may just be
   memorizing noise.

**Caveat:** none of this replaces a capacity-matched ablation (same
parameter count as QLayer but without the phase mechanism) if you want to
attribute a QLayer win specifically to the interference idea rather than to
"more parameters."

## 14 — Windowed (smoothed) correlation, robustness to local noise

Single-nucleotide-resolution Pearson/Spearman r can be dominated by
position-level count noise even when a model gets the broad local trend
right. This section re-computes Pearson r after smoothing both prediction
and target with a moving-average window of size `n` (a "n-window"), for a
few window sizes, purely as an additional read on the same predictions — it
does not retrain or change any model.

**What to look for:**
- If r increases substantially as the window grows (e.g. n=1 → n=25), the
  model is getting the coarse local shape right but not exact single-position
  peaks — a common and often acceptable limitation for profile models.
- If r barely changes with smoothing, the raw per-position result already
  reflects the model's real accuracy (little single-position noise to
  average out).
- Compare how baseline/Standard/QLayer respond differently to smoothing —
  a model that only catches up to another under heavy smoothing is weaker
  at fine-grained localization even if broad-shape performance looks similar.

In [ ]:
import torch.nn.functional as F


def smooth_last_dim(x: torch.Tensor, n_window: int) -> torch.Tensor:
    """Moving-average smoothing along the last dim with window size n_window.

    Purely for analysis — does not affect any model or training.
    """
    if n_window <= 1:
        return x
    pad = n_window // 2
    kernel = torch.ones(1, 1, n_window, device=x.device, dtype=x.dtype) / n_window
    x_padded = F.pad(x.unsqueeze(1), (pad, pad), mode="replicate")
    smoothed = F.conv1d(x_padded, kernel).squeeze(1)
    return smoothed[..., : x.shape[-1]]


@torch.no_grad()
def evaluate_pearson_windowed(model, dataloader, device, n_window):
    model.eval()
    corrs = []
    for batch in dataloader:
        seq    = batch["sequence"].to(device)
        signal = batch["signal"].to(device)
        pred, _ = model(seq)
        target = torch.log1p(signal)

        p = smooth_last_dim(pred.squeeze(1), n_window)
        t = smooth_last_dim(target.squeeze(1), n_window)
        pz = p - p.mean(-1, keepdim=True)
        tz = t - t.mean(-1, keepdim=True)
        r = (pz * tz).sum(-1) / (pz.norm(dim=-1) * tz.norm(dim=-1) + 1e-8)
        corrs.extend(r.cpu().float().numpy().tolist())
    return float(np.mean(corrs))


window_sizes = [1, 5, 10, 25, 50]

print(f"{'n_window':>10s}  {'Baseline':>10s}  {'Standard':>10s}  {'QLayer':>10s}")
print("-" * 46)
for n_window in window_sizes:
    r_base = evaluate_pearson_windowed(model_base, test_loader, device, n_window)
    r_std  = evaluate_pearson_windowed(model_std,  test_loader, device, n_window)
    r_ql   = evaluate_pearson_windowed(model_ql,   test_loader, device, n_window)
    print(f"{n_window:>10d}  {r_base:>10.4f}  {r_std:>10.4f}  {r_ql:>10.4f}")

## 15 — Integrated Gradients (sequence attribution)

Integrated Gradients (IG) attributes the model's predicted total signal back
to individual input positions — a per-nucleotide saliency map showing *which
parts of the sequence* drove the prediction, as opposed to Sections 1-14
which only look at aggregate accuracy.

**Gotcha:** `GlobalCLIPStandardModel.forward` / `GlobalCLIPQLayerModel.forward`
wrap the frozen PARNET backbone call in `torch.no_grad()`, which blocks
gradients from reaching the input sequence. `forward_with_grad` below
re-implements the identical forward computation *without* that no_grad block
(the backbone's weights stay frozen — `requires_grad=False` — only the
input-side gradient path is restored), purely for this attribution analysis;
it does not change the trained models.

Baseline: an all-zero one-hot vector ("no sequence"), the standard choice for
IG on one-hot categorical inputs. Because both input and baseline are
one-hot/zero, the attribution collapses to one value per position (only the
base actually present contributes, since `input - baseline = 0` for every
other channel by construction).

**What to look for:**
- Peaks in the IG track that line up with high true/predicted signal = the
  model is attributing importance to plausible positions, not spreading
  credit arbitrarily.
- Large attribution far from any real signal, or a flat/near-zero IG track
  everywhere, is a red flag that the model isn't learning position-specific
  sequence features.
- Compare Standard vs. QLayer on the same sequence: very different
  attribution patterns suggest the two architectures pick up on different
  signals — worth a deeper look if QLayer's phase story (Sections 5-7) is
  meant to be taken literally.

In [ ]:
def forward_with_grad(model, seq_onehot):
    """Re-run the model's forward pass without the internal no_grad on the
    backbone, so gradients can flow back to seq_onehot (needed for
    Integrated Gradients). Backbone weights stay frozen; only the
    input-side gradient path is restored. Analysis-only, doesn't mutate model.
    """
    x = model.backbone.stem(seq_onehot)
    x = model.backbone.body(x)
    if hasattr(model.backbone, "projection"):
        x = model.backbone.projection(x)
    embedding = x
    rbp_tracks = model.backbone.head.head_target.pointwise_conv(x)

    alpha = model.mix_coeff(embedding)
    scale = model.log_scale.exp()
    scaled = rbp_tracks * scale[None, :, None]

    if hasattr(model, "qlayer"):
        interference = model.qlayer(scaled, alpha)
        pred = model.cnn(interference)
    else:
        pred = (scaled * alpha[:, :, None]).sum(1, keepdim=True)
    return pred, alpha


def integrated_gradients(model, seq_onehot, steps=50, batch=10):
    """IG attribution of total predicted signal w.r.t. the input sequence.

    seq_onehot: (1, 4, L). Returns (L,) attribution per position.
    """
    baseline = torch.zeros_like(seq_onehot)
    alphas = torch.linspace(0, 1, steps, device=seq_onehot.device).view(steps, 1, 1)
    interpolated = baseline + alphas * (seq_onehot - baseline)   # (steps, 4, L)

    grads = []
    for i in range(0, steps, batch):
        chunk = interpolated[i:i + batch].clone().detach().requires_grad_(True)
        pred, _ = forward_with_grad(model, chunk)
        target = pred.sum()
        g, = torch.autograd.grad(target, chunk)
        grads.append(g.detach())
    avg_grad = torch.cat(grads, dim=0).mean(0)              # (4, L)
    ig = (seq_onehot.squeeze(0) - baseline.squeeze(0)) * avg_grad
    return ig.sum(0).cpu().numpy()                           # (L,)


# Reuse the same high-signal example selected in Section 4
ig_sample = test_ds[top_indices[0]]
ig_seq    = ig_sample["sequence"].unsqueeze(0).to(device)
ig_signal = ig_sample["signal"].squeeze().numpy()

logger.info("Computing Integrated Gradients for Standard and QLayer...")
ig_std = integrated_gradients(model_std, ig_seq)
ig_ql  = integrated_gradients(model_ql,  ig_seq)

fig, axes = plt.subplots(3, 1, figsize=(14, 7), sharex=True)
pos = np.arange(600)
axes[0].fill_between(pos, ig_signal, color="black", linewidth=0, alpha=0.8)
axes[0].set_title(f"Ground truth signal (test sample {top_indices[0]})")
axes[1].plot(pos, ig_std, color="steelblue", linewidth=1)
axes[1].set_title("Integrated Gradients — Standard")
axes[2].plot(pos, ig_ql, color="darkorange", linewidth=1)
axes[2].set_title("Integrated Gradients — QLayer")
axes[2].set_xlabel("Position")
plt.tight_layout()
fig.savefig(FILEPATHS.analysis_dir / "integrated_gradients_example.png", dpi=120, bbox_inches="tight")
plt.show()